# 01 — Pipeline de datos: Extracción, Carga y Análisis SQL
Este notebook ejecuta el pipeline completo del proyecto para las 18 series del Banco Central de Chile:
1. **Extracción y Carga:** Petición a la API SieteRestWS y persistencia idempotente en SQLite.
2. **Exploración Relacional:** Verificación del esquema `temas`, `series` y `observaciones`.
3. **Análisis Macro con SQL:** JOINs, agregaciones mensuales y variación interanual con funciones de ventana (`LAG`).

In [ ]:
import sys
from pathlib import Path
import pandas as pd
from sqlalchemy import text

# Permitir importaciones desde la carpeta src/
sys.path.append(str(Path.cwd().parent / "src"))

import config
import carga_db

# Fijar MODO DEMO en False si ya cuentas con tu .env configurado
DEMO = False
FECHA_CORTE = "2026-08-31"

# 1. Ejecución del pipeline de extracción y carga
print(f"Iniciando pipeline a SQLite (hasta: {FECHA_CORTE})...")
carga_db.main(demo=DEMO, hasta=FECHA_CORTE)

In [ ]:
# Conexión SQLAlchemy a la base de datos coyuntura.db
engine = carga_db.obtener_engine()

# Verificar las 3 tablas relacionales del proyecto
with engine.connect() as conn:
    tablas = pd.read_sql(text("SELECT name FROM sqlite_master WHERE type IN ('table','view')"), conn)

tablas

In [ ]:
# Catálogo relacional cruzando las 18 series con sus 4 temas oficiales
consulta_join = text("""
SELECT 
    s.id_serie, 
    s.nombre, 
    t.tema, 
    s.frecuencia, 
    s.unidad
FROM series s
JOIN temas t ON t.id_tema = s.id_tema
ORDER BY t.tema, s.nombre;
""")

with engine.connect() as conn:
    catalogo = pd.read_sql(consulta_join, conn)

catalogo

In [ ]:
# Resumen de cobertura temporal y recuento de observaciones por serie
consulta_groupby = text("""
SELECT 
    t.tema,
    s.nombre AS serie,
    MIN(o.fecha) AS primera_obs,
    MAX(o.fecha) AS ultima_obs,
    COUNT(*) AS total_observaciones
FROM observaciones o
JOIN series s ON s.id_serie = o.id_serie
JOIN temas t ON t.id_tema = s.id_tema
GROUP BY t.tema, s.nombre
ORDER BY t.tema, s.nombre;
""")

with engine.connect() as conn:
    cobertura = pd.read_sql(consulta_groupby, conn)

cobertura

In [ ]:
# Variación interanual (% 12 meses) procesando datos diarios a nivel mensual
consulta_ventana = text("""
WITH mensual AS (
    SELECT 
        o.id_serie,
        strftime('%Y-%m', o.fecha) AS mes,
        AVG(o.valor) AS valor_promedio
    FROM observaciones AS o
    GROUP BY o.id_serie, mes
),
variaciones AS (
    SELECT 
        m.id_serie,
        m.mes,
        m.valor_promedio,
        LAG(m.valor_promedio, 12) OVER (
            PARTITION BY m.id_serie 
            ORDER BY m.mes
        ) AS valor_hace_12m,
        100.0 * (m.valor_promedio / LAG(m.valor_promedio, 12) OVER (
            PARTITION BY m.id_serie 
            ORDER BY m.mes
        ) - 1.0) AS var_interanual_pct
    FROM mensual AS m
)
SELECT 
    t.tema,
    s.nombre AS serie,
    v.mes,
    ROUND(v.valor_promedio, 2) AS valor_mensual,
    ROUND(v.var_interanual_pct, 2) AS var_12m_pct
FROM variaciones AS v
JOIN series AS s ON s.id_serie = v.id_serie
JOIN temas AS t ON t.id_tema = s.id_tema
ORDER BY t.tema, s.nombre, v.mes;
""")

with engine.connect() as conn:
    df_interanual = pd.read_sql(consulta_ventana, conn)

# Mostrar muestra con variaciones interanuales calculadas
df_interanual.dropna(subset=["var_12m_pct"]).head(15)